<!-- nav -->
[← 5 · The shape of a population](05_The_Shape_Of_A_Population.ipynb) · [Home](../../README.md) · [7 · Generating test data →](07_Generating_Test_Data.ipynb)

# 6 · Drawing values

**The problem.** A believable senior isn't just "name: senior": they have an age, around 70, never under 65. A pattern
needs **distributions of values**, and they too must stay constraints: the same pattern checks existing contacts.

A distribution binds a **symbol** to a value drawn from it, within its body. The body's equality, `person.age == age`,
is what sets the property.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints, Distributions, Predicates, Validators } from "@mbse/patterns";
import { Contact, Directory, book, listed } from "./toolkit.ts";

const person = E.variable("person");

## 06.1 · a distribution of values

`Normal`, `Uniform`, `Poisson`, `Geometric` and `Categorical` are built like any term. `.requires(...)` calls a function
with the variable of the distribution's symbol; the outer names come from its closure:

In [2]:
const Age = new Distributions.Normal.Builder().symbol("age").mean(70n).deviation(8n).rounded()
  .requires((age) => person.age.eq(age)).create();
console.log(Age.symbol, Age.body.name);

age eq


## 06.2 · as a constraint, a distribution checks its support

Checking data, a distribution takes the value its body equates the symbol with, its **witness** (`person.age`, here),
and holds when that value is in the distribution's support and the body holds. A rounded normal's support is every int,
so it checks that the age is an int. A uniform's support is its range:

In [3]:
const Adults = new Predicates.OfPredicate.Builder().name("Adults").symbols({ person: Contact })
  .requires(new Distributions.Uniform.Builder().symbol("age").low(18n).high(64n).requires((age) => person.age.eq(age)).create()).create();
const store = book();
listed(store, store.Contact().name("Ann").age(34n).create(), store.Contact().name("Kid").age(9n).create(),
  store.Contact().name("Bob").create());
console.log(Validators.Validate(store, [Adults]).Reachable(Directory, store.singleton("book.Directory")));

[
  "person=Contact#2: 'Adults' does not hold",
  "person=Contact#3: 'Adults' is unknown"
]


The kid is outside [18, 64]. Bob has no age, so the uniform has no witness: unknown.

## 06.3 · schemas are checked before any data is read

A distribution's values have a schema, and so does a property. A normal gives floats unless it's rounded; an age is an
int. `Constraints.check` says so before any data is touched:

In [4]:
const Wrong = new Predicates.OfPredicate.Builder().name("Wrong").symbols({ person: Contact })
  .requires(new Distributions.Normal.Builder().symbol("age").mean(70n).deviation(8n).requires((age) => person.age.eq(age)).create()).create();
try {
  Constraints.check([Wrong]);
} catch (error) {
  console.log(String(error));
}

ValueError: predicate 'Wrong': person.age is int, but its distribution gives float


## 06.4 · choices of distributions

Arms can hold distributions, so a mixture is a choices. Here a senior's age is a normal restricted to 65 and over, and an
adult's is uniform:

In [5]:
const HasName = new Predicates.OfPredicate.Builder().name("HasName").symbols({ person: Contact })
  .parameters((x) => x.name("name")).requires(person.name.eq(E.variable("name"))).create();
const People = new Predicates.OfPredicate.Builder().name("People").symbols({ person: Contact }).requires(
  new Distributions.Choices.Builder().arms(
    (a) => a.weight(3).requires(HasName.call(person, "senior"), Age, person.age.ge(65n)),
    (a) => a.weight(7).requires(HasName.call(person, "adult"), new Distributions.Uniform.Builder().symbol("age")
      .low(18n).high(64n).requires((age) => person.age.eq(age)).create()),
  ).decreasing().create()).create();
console.log(Constraints.check([People]).predicates[0].name);

People


The next case study generates contacts from `People`.

## Why it works this way

- **Values reach properties only through equalities.** The body says where a drawn value goes, and the same equality,
  read the other way, gives the witness a validator checks; with no witness, the distribution is unknown.
- **Rounded, not truncated.** `.rounded()` rounds half up, so ints come out of a normal centred where you put it.

The design: [CAP-072: distributions in predicates](../../../mbse-basis/docs/capabilities/CAP-072.md); [CAP-029: three-valued constraint checking](../../../mbse-basis/docs/capabilities/CAP-029.md).

<!-- nav -->
[← 5 · The shape of a population](05_The_Shape_Of_A_Population.ipynb) · [Home](../../README.md) · [7 · Generating test data →](07_Generating_Test_Data.ipynb)